# Analyse Exploratoire du Projet Assistant Restauration Rapide
Ce notebook explore les donnees du projet : FAQ, menus, embeddings.
Objectif : comprendre la structure des donnees, la distribution des categories,
et evaluer la qualite des representations vectorielles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
sns.set_theme()
%matplotlib inline

## 1. Chargement et exploration du dataset FAQ

In [ ]:
DATA_DIR = os.path.join('..', 'data')
faq_path = os.path.join(DATA_DIR, 'faq_dataset.csv')
df = pd.read_csv(faq_path)
df.head()

In [ ]:
df.info()

## 2. Distribution des categories

In [ ]:
cat_counts = df['categorie'].value_counts()
cat_counts

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.barplot(x=cat_counts.index, y=cat_counts.values, hue=cat_counts.index, palette='Set2', legend=False)
plt.title('Distribution des categories de questions FAQ')
plt.xlabel('Categorie')
plt.ylabel('Nombre de questions')
for i, v in enumerate(cat_counts.values):
    ax.text(i, v + 0.3, str(v), ha='center', fontweight='bold')
plt.show()

## 3. Analyse des longueurs de questions et reponses

In [ ]:
df['longueur_question'] = df['question'].str.len()
df['longueur_reponse'] = df['reponse'].str.len()
df[['categorie', 'question', 'longueur_question', 'longueur_reponse']].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['longueur_question'], bins=20, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution des longueurs de questions')
axes[0].set_xlabel('Nombre de caracteres')
sns.histplot(df['longueur_reponse'], bins=20, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Distribution des longueurs de reponses')
axes[1].set_xlabel('Nombre de caracteres')
plt.tight_layout()
plt.show()

In [ ]:
print('Statistiques questions :')
print(df['longueur_question'].describe().round(1))
print()
print('Statistiques reponses :')
print(df['longueur_reponse'].describe().round(1))

## 4. Analyse TF-IDF des questions

In [ ]:
FRENCH_STOPS = ['le', 'la', 'les', 'du', 'des', 'un', 'une', 'de', 'ce',
                'ces', 'mon', 'ton', 'son', 'ma', 'ta', 'sa', 'mes', 'tes',
                'ses', 'nos', 'vos', 'leurs', 'je', 'tu', 'il', 'elle', 'on',
                'nous', 'vous', 'ils', 'elles', 'et', 'ou', 'ou', 'qui', 'que',
                'quoi', 'dont', 'au', 'aux', 'est', 'sont', 'avez', 'ont',
                'fait', 'peut', 'dans', 'sur', 'avec', 'pour', 'par', 'pas',
                'plus', 'tres', 'bien', 'si', 'non', 'oui']
vectorizer = TfidfVectorizer(stop_words=FRENCH_STOPS)
tfidf_matrix = vectorizer.fit_transform(df['question'])
print(f'Matrice TF-IDF : {tfidf_matrix.shape}')
print(f'Nombre de termes : {len(vectorizer.get_feature_names_out())}')

In [ ]:
avg_tfidf = np.array(tfidf_matrix.mean(axis=0)).flatten()
top_idx = avg_tfidf.argsort()[-15:][::-1]
top_words = [(vectorizer.get_feature_names_out()[i], avg_tfidf[i]) for i in top_idx]
print('Top 15 mots les plus importants (TF-IDF moyen) :')
for word, score in top_words:
    print(f'  {word}: {score:.4f}')

In [ ]:
sim_matrix = cosine_similarity(tfidf_matrix)
np.fill_diagonal(sim_matrix, 0)
idx_sim = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
print(f'Questions les plus similaires (TF-IDF, score={sim_matrix[idx_sim]:.3f}) :')
print(f'  Q1: {df.iloc[idx_sim[0]]["question"][:80]}...')
print(f'  Q2: {df.iloc[idx_sim[1]]["question"][:80]}...')

## 5. Chargement et exploration des menus JSON

In [ ]:
for brand_file in ['quick_menu.json', 'bk_menu.json', 'mcdo_menu.json', 'kfc_menu.json']:
    path = os.path.join(DATA_DIR, brand_file)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    n_items = sum(len(items) for items in data['menus'].values())
    n_cats = len(data['menus'])
    n_promos = len(data.get('promotions', []))
    print(f'{brand_file.replace("_menu.json",""):>8} : {n_cats} categories, {n_items} produits, {n_promos} promos')

## 6. Analyse des prix par enseigne

In [ ]:
prices = {}
for brand_file in ['quick_menu.json', 'bk_menu.json', 'mcdo_menu.json', 'kfc_menu.json']:
    path = os.path.join(DATA_DIR, brand_file)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    brand_name = brand_file.replace('_menu.json', '')
    p = []
    for items in data['menus'].values():
        for item in items:
            if item.get('prix'):
                p.append(float(item['prix']))
    prices[brand_name] = p
fig, ax = plt.subplots(figsize=(12, 5))
parts = ax.boxplot(prices.values(), tick_labels=prices.keys(), patch_artist=True)
colors = ['#ff9999','#66b3ff','#99ff99','#ffcc99']
for patch, color in zip(parts['boxes'], colors):
    patch.set_facecolor(color)
ax.set_title('Distribution des prix par enseigne')
ax.set_ylabel('Prix (EUR)')
ax.grid(axis='y', alpha=0.3)
plt.show()

## 7. Visualisation PCA des embeddings TF-IDF

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(tfidf_matrix.toarray())
print(f'Variance expliquee par PC1+PC2 : {pca.explained_variance_ratio_.sum():.1%}')
plt.figure(figsize=(10, 7))
scatter = plt.scatter(coords[:, 0], coords[:, 1],
                      c=pd.Categorical(df['categorie']).codes,
                      cmap='Set2', s=80, alpha=0.8)
plt.title('Projection PCA des questions FAQ (TF-IDF)')
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
cbar = plt.colorbar(scatter, ticks=range(len(cat_counts)))
cbar.set_ticklabels(cat_counts.index.tolist())
plt.tight_layout()
plt.show()

In [ ]:
print('Analyse exploratoire terminee.')
print(f'Total questions FAQ : {len(df)}')
print(f'Total categories : {df["categorie"].nunique()}')
print(f'Enseignes gerees : Quick, Burger King, McDonald\'s, KFC')